# Great Expectations Data Quality Validation

## 1. Great Expectations Context Setup (TE)

In [ ]:
# Cell 1 - Imports

import pandas as pd
import great_expectations as gx
from great_expectations import expectations as gxe

# added for Dagster

from google.cloud import bigquery
client = bigquery.Client(project="<your_gcp_project_id>")

In [ ]:
# Cell 2 - Get context

context = gx.get_context()
context

## 2. Fact Table Validation: fct_order_items (TEBE)

In [ ]:
# Cell 3 - Get data

query = """
SELECT *
FROM `<your_gcp_project_id>.olist_analytics.fct_order_items`
LIMIT 10000
"""

df_fct_order_items = pd.read_gbq(query, project_id="<your_gcp_project_id>")
df_fct_order_items.head()

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="<your_gcp_project_id>")

query = """
SELECT *
FROM `<your_gcp_project_id>.olist_analytics.fct_order_items`
LIMIT 10000
"""

df_fct_order_items = client.query(query).to_dataframe()
df_fct_order_items.head()

In [ ]:
# Cell 4 - Create Great Expectations DataFrame

# Create datasource
data_source = context.data_sources.add_pandas(name="pandas_source")

# Add dataframe as asset
data_asset = data_source.add_dataframe_asset(name="fct_order_items_df")

# Define batch
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch")

# Get batch (this is your GX object)
batch = batch_definition.get_batch(batch_parameters={"dataframe": df_fct_order_items})

batch

In [ ]:
# Cell 5A - create expectation suite

import great_expectations as gx
from great_expectations.core.expectation_suite import ExpectationSuite

suite_name = "fct_order_items_suite"

try:
    suite = context.suites.get(name=suite_name)
    print(f"Loaded existing suite: {suite_name}")
except Exception:
    suite = ExpectationSuite(name=suite_name)
    context.suites.add(suite)
    print(f"Created new suite: {suite_name}")

In [ ]:
# Cell 5B - create validator

validator = context.get_validator(
    batch=batch,
    expectation_suite_name=suite_name
)

validator

In [ ]:
# Cell 6 - structural checks

validator.expect_column_values_to_not_be_null("order_id")
validator.expect_column_values_to_not_be_null("order_item_id")
validator.expect_column_values_to_not_be_null("product_id")
validator.expect_column_values_to_not_be_null("seller_id")
validator.expect_column_values_to_not_be_null("customer_id")

In [ ]:
# Cell 7 - business checks

validator.expect_column_values_to_be_between("price", min_value=0)
validator.expect_column_values_to_be_between("freight_value", min_value=0)
validator.expect_column_values_to_be_between("gross_item_value", min_value=0)
validator.expect_table_row_count_to_be_between(min_value=1)

In [ ]:
# Cell 8 - update expectation suite

suite = validator.get_expectation_suite()
context.suites.add_or_update(suite)

print("Expectation suite updated.")

In [ ]:
# Cell 9 - validate data

results = validator.validate()
results

In [ ]:
# Cell 10 - validate data

results = validator.validate()
results

In [ ]:
# Cell 11 - validate data with more details

suite = validator.get_expectation_suite()
context.suites.add_or_update(suite)
print("Expectation suite updated.")

results = validator.validate()
print("Overall success:", results.success)
print("Expectations run:", len(results.results))
results

In [ ]:
# Cell 12 - Executive Data Quality Summary (TEBE)

import pandas as pd

summary_rows = []

for r in results.results:
    config = r.expectation_config

    try:
        config_dict = config.to_json_dict()
    except Exception:
        config_dict = {}

    expectation = (
        config_dict.get("type")
        or getattr(config, "type", None)
        or getattr(config, "expectation_type", None)
        or "unknown_expectation"
    )

    kwargs = config_dict.get("kwargs") or getattr(config, "kwargs", {}) or {}
    column = kwargs.get("column", "table")

    summary_rows.append({
        "expectation": expectation,
        "column": column,
        "success": r.success
    })

summary_df = pd.DataFrame(summary_rows)

summary_df

In [ ]:
# Cell 13 - Validation Summary Stats (TEBE)

total_checks = len(summary_df)
passed_checks = int(summary_df["success"].sum())
failed_checks = total_checks - passed_checks
success_rate = passed_checks / total_checks if total_checks > 0 else 0

print(f"Total Checks: {total_checks}")
print(f"Passed: {passed_checks}")
print(f"Failed: {failed_checks}")
print(f"Success Rate: {success_rate:.2%}")

In [ ]:
# Cell 14 - Failed Checks Only (TEBE)

summary_df[summary_df["success"] == False]

In [ ]:
# Debug cell (TE)

first_result = results.results[0]
print(type(first_result))
print(type(first_result.expectation_config))
print(dir(first_result.expectation_config))

## 3. Dimension Table Validation: dim_customers (TE)

In [ ]:
# 3.1 - Load table

query = """
SELECT *
FROM `<your_gcp_project_id>.olist_analytics.dim_customers`
"""

df_dim_customers = client.query(query).to_dataframe()
df_dim_customers.head()

In [ ]:
# 3.2 - Add dataframe asset

data_source = context.data_sources.add_or_update_pandas(name="pandas_source")

data_asset = data_source.add_dataframe_asset(name="dim_customers_df")

batch_definition = data_asset.add_batch_definition_whole_dataframe("batch")

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df_dim_customers}
)

batch

In [ ]:
# 3.3 - Create/load suite

from great_expectations.core.expectation_suite import ExpectationSuite

suite_name = "dim_customers_suite"

try:
    suite = context.suites.get(name=suite_name)
    print(f"Loaded existing suite: {suite_name}")
except Exception:
    suite = ExpectationSuite(name=suite_name)
    context.suites.add(suite)
    print(f"Created new suite: {suite_name}")

In [ ]:
# 3.4 - Create validator

validator = context.get_validator(
    batch=batch,
    expectation_suite_name=suite_name
)

validator

In [ ]:
# 3.5 - Expectations

validator.expect_column_values_to_not_be_null("customer_id")
validator.expect_column_values_to_be_unique("customer_id")
validator.expect_column_values_to_not_be_null("customer_state")
validator.expect_table_row_count_to_be_between(min_value=1)

In [ ]:
# 3.6 - Save/update suite and validate

suite = validator.get_expectation_suite()
context.suites.add_or_update(suite)

results = validator.validate()
print("Overall success:", results.success)
print("Expectations run:", len(results.results))
results

In [ ]:
# 3.7 - reuse cell 12 - Executive Data Quality Summary (TEBE)

import pandas as pd

summary_rows = []

for r in results.results:
    config = r.expectation_config

    try:
        config_dict = config.to_json_dict()
    except Exception:
        config_dict = {}

    expectation = (
        config_dict.get("type")
        or getattr(config, "type", None)
        or getattr(config, "expectation_type", None)
        or "unknown_expectation"
    )

    kwargs = config_dict.get("kwargs") or getattr(config, "kwargs", {}) or {}
    column = kwargs.get("column", "table")

    summary_rows.append({
        "expectation": expectation,
        "column": column,
        "success": r.success
    })

summary_df = pd.DataFrame(summary_rows)

summary_df

## 3. Dimension Table Validation (TE)

### 3.1–3.4 Parameterized Validation for dim_customers, dim_products, dim_sellers, dim_dates (TE)

In [ ]:
# Cell section 3.1 to 3.4

# Dimension Table Validation (TE) - Parameterized Execution for Remaining 4 Tables

import pandas as pd
from great_expectations.core.expectation_suite import ExpectationSuite

# Reuse or create the pandas datasource
data_source = context.data_sources.add_or_update_pandas(name="pandas_source")

# Rules for remaining 4 dimension tables
validation_rules = {
    "dim_customers": {
        "suite_name": "dim_customers_suite",
        "asset_name": "dim_customers_df",
        "query": """
            SELECT *
            FROM `<your_gcp_project_id>.olist_analytics.dim_customers`
        """,
        "expectations": {
            "not_null": ["customer_id", "customer_state"],
            "unique": ["customer_id"],
            "between": {},
            "row_count_min": 1,
        },
        "audience": "TE",
    },
    "dim_products": {
        "suite_name": "dim_products_suite",
        "asset_name": "dim_products_df",
        "query": """
            SELECT *
            FROM `<your_gcp_project_id>.olist_analytics.dim_products`
        """,
        "expectations": {
            "not_null": ["product_id"],
            "unique": ["product_id"],
            "between": {
                "product_weight_g": {"min_value": 0},
                "product_length_cm": {"min_value": 0},
                "product_height_cm": {"min_value": 0},
                "product_width_cm": {"min_value": 0},
            },
            "row_count_min": 1,
        },
        "audience": "TE",
    },
    "dim_sellers": {
        "suite_name": "dim_sellers_suite",
        "asset_name": "dim_sellers_df",
        "query": """
            SELECT *
            FROM `<your_gcp_project_id>.olist_analytics.dim_sellers`
        """,
        "expectations": {
            "not_null": ["seller_id", "seller_state"],
            "unique": ["seller_id"],
            "between": {},
            "row_count_min": 1,
        },
        "audience": "TE",
    },
    "dim_dates": {
        "suite_name": "dim_dates_suite",
        "asset_name": "dim_dates_df",
        "query": """
            SELECT *
            FROM `<your_gcp_project_id>.olist_analytics.dim_dates`
        """,
        "expectations": {
            "not_null": ["date_id"],
            "unique": ["date_id"],
            "between": {
                "month": {"min_value": 1, "max_value": 12},
                "day_of_week": {"min_value": 1, "max_value": 7},
            },
            "row_count_min": 1,
        },
        "audience": "TE",
    },
}


def run_validation_for_table(context, client, data_source, table_name, config):
    """
    Run Great Expectations validation for one table using parameterized rules.

    Args:
        context: Great Expectations context.
        client: BigQuery client.
        data_source: Great Expectations pandas datasource.
        table_name: Logical name of the table.
        config: Validation configuration for the table.

    Returns:
        dict: Validation results and summary information for the table.
    """
    # Load table into pandas
    df = client.query(config["query"]).to_dataframe()

    # Add dataframe asset
    asset_name = config["asset_name"]

    try:
        data_asset = data_source.get_asset(asset_name)
    except Exception:
        data_asset = data_source.add_dataframe_asset(name=asset_name)

    # Batch definition
    try:
        batch_definition = data_asset.get_batch_definition("batch")
    except Exception:
        batch_definition = data_asset.add_batch_definition_whole_dataframe("batch")

    batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

    # Create or load expectation suite
    suite_name = config["suite_name"]

    try:
        context.suites.get(name=suite_name)
    except Exception:
        context.suites.add(ExpectationSuite(name=suite_name))

    # Validator
    validator = context.get_validator(
        batch=batch,
        expectation_suite_name=suite_name
    )

    # Optional: remove old expectations if rerunning a lot and wanting clean replacement
    # Comment this in ONLY if repeated reruns cause duplicated expectations in a suite.
    # validator.expectation_suite.expectations = []

    # Apply expectations
    for col in config["expectations"]["not_null"]:
        validator.expect_column_values_to_not_be_null(col)

    for col in config["expectations"]["unique"]:
        validator.expect_column_values_to_be_unique(col)

    for col, bounds in config["expectations"]["between"].items():
        validator.expect_column_values_to_be_between(col, **bounds)

    validator.expect_table_row_count_to_be_between(
        min_value=config["expectations"]["row_count_min"]
    )

    # Save/update suite
    suite = validator.get_expectation_suite()
    context.suites.add_or_update(suite)

    # Validate
    results = validator.validate()

    # Build detailed summary
    summary_rows = []
    for r in results.results:
        config_obj = r.expectation_config

        try:
            config_dict = config_obj.to_json_dict()
        except Exception:
            config_dict = {}

        expectation = (
            config_dict.get("type")
            or getattr(config_obj, "type", None)
            or getattr(config_obj, "expectation_type", None)
            or "unknown_expectation"
        )

        kwargs = config_dict.get("kwargs") or getattr(config_obj, "kwargs", {}) or {}
        column = kwargs.get("column", "table")

        summary_rows.append({
            "table_name": table_name,
            "audience": config["audience"],
            "expectation": expectation,
            "column": column,
            "success": r.success,
        })

    summary_df = pd.DataFrame(summary_rows)

    return {
        "table_name": table_name,
        "audience": config["audience"],
        "dataframe": df,
        "results": results,
        "summary_df": summary_df,
        "total_checks": len(summary_df),
        "passed_checks": int(summary_df["success"].sum()),
        "failed_checks": len(summary_df) - int(summary_df["success"].sum()),
        "success_rate": (
            int(summary_df["success"].sum()) / len(summary_df)
            if len(summary_df) > 0 else 0
        ),
    }


# Run validation for all remaining dimension tables
all_dimension_results = []

for table_name, config in validation_rules.items():
    result = run_validation_for_table(
        context=context,
        client=client,
        data_source=data_source,
        table_name=table_name,
        config=config,
    )
    all_dimension_results.append(result)

    print(
        f"{table_name}: "
        f"{result['passed_checks']}/{result['total_checks']} passed "
        f"({result['success_rate']:.2%}) | "
        f"Overall success = {result['results'].success}"
    )

# Master summary for the 4 dimension tables
dimension_master_summary = pd.DataFrame([
    {
        "table_name": r["table_name"],
        "audience": r["audience"],
        "total_checks": r["total_checks"],
        "passed_checks": r["passed_checks"],
        "failed_checks": r["failed_checks"],
        "success_rate": round(r["success_rate"], 4),
        "overall_success": r["results"].success,
    }
    for r in all_dimension_results
])

dimension_master_summary

### Note on Data Quality Handling (TE)

During validation, an issue was detected in the product dataset due to invalid numeric casting from empty string values. This was resolved upstream using a robust ingestion pattern:

- nullif(...) to handle empty strings
- safe_cast(...) to prevent pipeline failure

This ensures the data pipeline remains resilient to malformed source data.

## 4. Data Quality Summary for Trusted Analytics Delivery (TEBE)

In [ ]:
# cell section 4.1
 
# Combined GE Summary for Fact + Dimensions (TEBE)

fct_order_items_master = pd.DataFrame([{
    "table_name": "fct_order_items",
    "audience": "TEBE",
    "total_checks": len(summary_df),
    "passed_checks": int(summary_df["success"].sum()),
    "failed_checks": len(summary_df) - int(summary_df["success"].sum()),
    "success_rate": round(
        int(summary_df["success"].sum()) / len(summary_df), 4
    ) if len(summary_df) > 0 else 0,
    "overall_success": results.success,
}])

ge_master_summary = pd.concat(
    [fct_order_items_master, dimension_master_summary],
    ignore_index=True
)

ge_master_summary

In [ ]:
# Cell section 4.2

# Overall GE Validation Totals (TEBE)

total_tables = len(ge_master_summary)
tables_passed = int(ge_master_summary["overall_success"].sum())
tables_failed = total_tables - tables_passed

total_checks = int(ge_master_summary["total_checks"].sum())
passed_checks = int(ge_master_summary["passed_checks"].sum())
failed_checks = int(ge_master_summary["failed_checks"].sum())

print(f"Tables validated: {total_tables}")
print(f"Tables passed: {tables_passed}")
print(f"Tables failed: {tables_failed}")
print(f"Total checks run: {total_checks}")
print(f"Checks passed: {passed_checks}")
print(f"Checks failed: {failed_checks}")
print(f"Overall check pass rate: {passed_checks / total_checks:.2%}")